# Choose a provider dataset and retrieve a small selection

**Goal:** find an invented housing table, choose one area and year, and save a
reproducible dataset. You will use the same connector methods for other supported
providers; the available dimensions and selection codes come from each provider.

All titles, codes, places and values here are **synthetic teaching material**.
The Nomis responses come from an in-memory fixture, with no provider connection.
This is a lesson in choosing data, not evidence about housing.

## Setup

Install SSDU using the [installation guide](https://xangma.github.io/ssdu/start/install.html).
If downloading this notebook, also download `provider_selection_fixture.py` from
[this lesson's manual page](https://xangma.github.io/ssdu/start/provider-selection.html)
and keep both files in the same folder. In a repository checkout they are already
alongside one another. Start Jupyter in that folder and run each cell in order.

In [ ]:
from pathlib import Path

import pandas as pd
from provider_selection_fixture import teaching_client

from ssdu import Connector, Request

workspace = Path.cwd() / "provider-selection-work"
workspace.mkdir(exist_ok=True)
client = teaching_client(workspace)
provider = Connector("nomis", root=workspace / "saved-data", client=client)

## 1. Browse a catalogue

Search for a topic and inspect the returned titles and IDs. A title helps you
choose; the provider's ID identifies the dataset in later calls. `limit` bounds
the returned page. Providers can differ in how their catalogue is downloaded
and paginated; this is not a universal limit on upstream catalogue bytes.

In [ ]:
page = provider.list_datasets("housing", limit=5)
pd.DataFrame(page["items"])[["id", "title", "provider"]]

The teaching catalogue contains one result: `NM_999_1`, **Invented housing
counts**. In a larger catalogue, compare the titles before choosing a row.
If `next_offset` is present, use it as `offset` to read the next page.

In [ ]:
dataset_id = page["items"][0]["id"]
description = provider.describe_dataset(dataset_id)
print(description["title"])
pd.DataFrame(description["dimensions"])[["id", "label", "selector"]]

## 2. Choose dimension codes

The description lists `geography`, `time` and `measures`. A **dimension** is a way
to select part of a table. Browse its codes before writing a request: display
labels such as “Example North” are not necessarily valid selectors.

In [ ]:
areas = provider.list_dimension_values(dataset_id, "geography", limit=5)
years = provider.list_dimension_values(dataset_id, "time", limit=5)
measures = provider.list_dimension_values(dataset_id, "measures", limit=5)
for label, result in [("Areas", areas), ("Years", years), ("Measures", measures)]:
    print(label)
    print(pd.DataFrame(result["items"])[["value", "label"]].to_string(index=False))

Choose **Example North**, **2024**, and **Invented dwelling count** from the
returned values. Keep codes such as `001` as strings: leading zeroes matter.
Nomis uses `date` to select its `time` dimension. Other providers use their own
selector syntax; inspect their descriptions rather than copying Nomis codes.

In [ ]:
area = next(row["value"] for row in areas["items"] if row["label"] == "Example North")
year = next(row["value"] for row in years["items"] if row["label"] == "2024")
measure = measures["items"][0]["value"]
request = Request(
    provider="nomis",
    dataset_id=dataset_id,
    params={"geography": area, "date": year, "measures": measure},
    page_size=2,
    max_rows=10,
)
request.to_dict()

## 3. Review the plan before retrieving values

A plan checks the request against available metadata. It can retrieve metadata
when that metadata is not cached; it does not download observation values.
`max_rows=10` caps accepted source rows and `page_size=2` bounds each data page.
These bounds are safeguards, not an estimate of the selected table's size.
A valid plan does not prove that a selection contains observations.

In [ ]:
plan = provider.plan_fetch(request)
print("Selection:", plan["request"]["params"])
print("Limits:", plan["limits"])
print("Value requests while planning:", plan["provider_requests_for_values"])

## 4. Retrieve and interpret the saved table

The connector now acquires the selection, normalises it and verifies the saved
snapshot. SSDU keeps an observation table as well as the source information.
Here we show the columns needed to interpret this one result.

In [ ]:
dataset = provider.fetch_dataset(request)
observations = dataset.to_pandas("observations")
observations[["geography", "period", "measure", "value", "unit", "status"]]

The row is an **invented count of 12 dwellings**, for Example North (`001`) in
2024. It is a count, not a percentage. Before analysing real data, inspect units,
coverage, missing-value status, source definitions and licence conditions.
Tables from different providers are not automatically comparable.

The snapshot ID identifies the version SSDU saved. Record it with your analysis.

In [ ]:
snapshot_note = workspace / "snapshot-id.txt"
snapshot_note.write_text(dataset.snapshot_id, encoding="utf-8")
print("Saved snapshot:", dataset.snapshot_id)

## 5. Reopen in a later session

This cell is self-contained: after restarting the kernel, run it in the same
folder without repeating the preceding steps. It reads saved files and needs
no provider connection. Keep the `provider-selection-work` folder.

In [ ]:
from pathlib import Path

from ssdu import Store

workspace = Path.cwd() / "provider-selection-work"
saved_id = (workspace / "snapshot-id.txt").read_text(encoding="utf-8").strip()
reopened = Store(workspace / "saved-data").open_dataset(saved_id)
print("Reopened snapshot:", reopened.snapshot_id)
reopened.to_pandas("observations")[["geography", "period", "value"]]

## Check: replay the request and catalogue offline

Opening a snapshot above reads that exact saved version. Replaying the original
request below also uses retained evidence. `offline=True` fails if the needed
response is missing; it never fills the gap from a provider.

In [ ]:
offline_provider = Connector("nomis", root=workspace / "saved-data")
replayed_page = offline_provider.list_datasets("housing", limit=5, offline=True)
cached = offline_provider.fetch_dataset(request, offline=True)
assert replayed_page["items"] == page["items"]
assert cached.snapshot_id == reopened.snapshot_id == dataset.snapshot_id
assert cached.to_arrow().equals(reopened.to_arrow())
print("Catalogue and selected dataset replayed offline.")

## Next steps

For a live Nomis session, create `Connector("nomis", root="my-data")` without the
teaching client, browse the real catalogue and construct a new bounded request
from its dimensions. The invented ID in this lesson is not a live dataset.
Live calls use SSDU's cached, rate-limited transport. Consult the
[provider workflow](https://xangma.github.io/ssdu/guides/provider-workflow.html)
and [coverage matrix](https://xangma.github.io/ssdu/reference/sources.html) first.

To search a local metadata cache across providers, continue to
[discover datasets from metadata](https://xangma.github.io/ssdu/start/metadata-discovery.html).
To preserve several selections together, see
[transfer a collection](https://xangma.github.io/ssdu/start/quickstart.html).